# Carregando o dataset

In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("TimJaspersTue/RARE25-train", split='train')

README.md:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'TimJaspersTue/RARE25-train' is a gated dataset on the Hub. You must be authenticated to access it.

In [ ]:
!pip install albumentationsx

# Treinamento com VIT

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
from torch.utils.data import DataLoader
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Configurar o dispositivo (GPU ou CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

# 2. Carregar e dividir o dataset
ds = load_dataset("TimJaspersTue/RARE25-train", split='train')
ds_split = ds.train_test_split(test_size=0.2)
ds_train = ds_split['train']
ds_val = ds_split['test']

# 3. Definir as transformações de pré-processamento
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.RandomCrop(224, 224),
    A.SquareSymmetry(p=0.5),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# 4. **Função de Pré-processamento**
def preprocess_image(image, transforms):
    image_np = np.array(image.convert('RGB'))
    transformed = transforms(image=image_np)
    return transformed['image']

# 5. Criar uma função de colaboração (`collate_fn`)
def custom_collate_fn(batch):
    images = [preprocess_image(item['image'], train_transforms) for item in batch]
    labels = [item['label'] for item in batch]
    
    images = torch.stack(images, dim=0)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return images, labels

# 6. Criar os DataLoaders
train_loader = DataLoader(ds_train, batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(ds_val, batch_size=16, collate_fn=custom_collate_fn)

# 7. **Carregar o modelo ViT do `torchvision` e ajustar**
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# Congela os pesos de todas as camadas, exceto a de classificação (head)
for param in model.parameters():
    param.requires_grad = False

# Ajusta a camada final de classificação para o número de classes (2)
num_ftrs = model.heads.head.in_features
model.heads.head = nn.Linear(num_ftrs, 2)
model.to(device)

# 8. Definir a função de perda e o otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.heads.parameters(), lr=0.001)

# 9. Loop de treinamento
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    # 10. Loop de validação
    model.eval()
    total_val_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(ds_train)
    epoch_val_loss = total_val_loss / len(ds_val)
    accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, "
          f"Val Loss: {epoch_val_loss:.4f}, "
          f"Val Acc: {accuracy:.2f}%")

print("Treinamento concluído!")



In [ ]:
# 11. **Gera e exibe a matriz de confusão**
cm = confusion_matrix(all_labels, all_preds)
class_names = ['Classe 0', 'Classe 1']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Configurar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

# 2. Carregar dataset
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
ds_split = ds.train_test_split(test_size=0.2, seed=42)
ds_train, ds_val = ds_split["train"], ds_split["test"]

# 3. Transformações
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.RandomCrop(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# 4. Pré-processamento
def preprocess_image(image, transforms):
    image_np = np.array(image.convert("RGB"))
    transformed = transforms(image=image_np)
    return transformed["image"]

# 5. Collate fn
def custom_collate_fn(batch, transforms):
    images = [preprocess_image(item["image"], transforms) for item in batch]
    labels = [item["label"] for item in batch]
    return torch.stack(images, dim=0), torch.tensor(labels, dtype=torch.long)

train_loader = DataLoader(
    ds_train, batch_size=16, shuffle=True,
    collate_fn=lambda b: custom_collate_fn(b, train_transforms)
)
val_loader = DataLoader(
    ds_val, batch_size=16, shuffle=False,
    collate_fn=lambda b: custom_collate_fn(b, val_transforms)
)

# 6. Pesos de classe
labels_train = [item["label"] for item in ds_train]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_train),
    y=labels_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# 7. Carregar modelo ViT-B16
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# liberar último bloco + head
for name, param in model.named_parameters():
    if "encoder.layers.encoder_layer_11" in name or "heads" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# ajustar camada final
num_ftrs = model.heads.head.in_features
model.heads.head = nn.Linear(num_ftrs, 2)
model.to(device)

# 8. Loss e otimizador
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# 9. Treino
num_epochs = 15
best_acc = 0.0

for epoch in range(num_epochs):
    # --- treino ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    # --- validação ---
    model.eval()
    total_val_loss = 0.0
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(ds_train)
    epoch_val_loss = total_val_loss / len(ds_val)
    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, "
          f"Val Loss: {epoch_val_loss:.4f}, "
          f"Val Acc: {accuracy:.2f}%")

    # salvar melhor modelo
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), "best_vit_b16.pth")
        print(">> Novo melhor modelo salvo!")

# 10. Matriz de confusão
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Classe 0", "Classe 1"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - ViT-B16")
plt.show()

print("Treinamento concluído!")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from PIL import Image

# ------------------------
# DATA AUGMENTATION
# ------------------------
train_transforms = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor()
])

augment_class1 = T.Compose([
    T.Resize((224, 224)),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAffine(degrees=15, translate=(0.1,0.1), scale=(0.9, 1.1)),
    T.RandomVerticalFlip(),
    T.ToTensor()
])

val_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

# ------------------------
# DATASET + SAMPLER
# ------------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
ds_split = ds.train_test_split(test_size=0.2, seed=42)
ds_train, ds_val = ds_split["train"], ds_split["test"]

targets = [sample['label'] for sample in ds_train]
class_sample_counts = np.bincount(targets)
weights = 1.0 / class_sample_counts
sample_weights = [weights[t] for t in targets]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

def custom_collate_fn(batch, transform):
    images, labels = [], []
    for item in batch:
        img = item['image']
        lbl = item['label']
        if lbl == 1 and transform == train_transforms and np.random.rand() > 0.5:
            img = augment_class1(img)
        else:
            img = transform(img)
        images.append(img)
        labels.append(lbl)
    
    return torch.stack(images), torch.tensor(labels)

train_loader = DataLoader(
    ds_train,
    batch_size=16,
    sampler=sampler,
    collate_fn=lambda b: custom_collate_fn(b, train_transforms)
)

val_loader = DataLoader(
    ds_val,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda b: custom_collate_fn(b, val_transforms)
)

# ------------------------
# MODELO VIT-B16
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Carrega o modelo ViT-B16 do torchvision
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# Congela todos os parâmetros por padrão
for param in model.parameters():
    param.requires_grad = False

# Libera o último bloco do encoder e o classificador (head)
for name, param in model.named_parameters():
    if "encoder.layers.encoder_layer_11" in name or "heads" in name:
        param.requires_grad = True

# O classificador do ViT já é ajustado automaticamente pela linha acima,
# mas podemos redefini-lo para maior clareza e controle
num_ftrs = model.heads.head.in_features
model.heads.head = nn.Linear(num_ftrs, 2)
model.to(device)

# ------------------------
# LOSS + OPTIMIZER
# ------------------------
labels_train = [sample['label'] for sample in ds_train]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_train),
    y=labels_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# ------------------------
# LOOP DE TREINO
# ------------------------
EPOCHS = 15
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(train_loader):.4f}")

# ------------------------
# AVALIAÇÃO - MATRIZ DE CONFUSÃO
# ------------------------
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Classe 0","Classe 1"], yticklabels=["Classe 0","Classe 1"])
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - ViT-B16")
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from PIL import Image

# ------------------------
# DATA AUGMENTATION
# ------------------------
train_transforms = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor()
])

augment_class1 = T.Compose([
    T.Resize((224, 224)),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAffine(degrees=15, translate=(0.1,0.1), scale=(0.9, 1.1)),
    T.RandomVerticalFlip(),
    T.ToTensor()
])

val_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

# ------------------------
# DATASET + SAMPLER
# ------------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
ds_split = ds.train_test_split(test_size=0.2, seed=42)
ds_train, ds_val = ds_split["train"], ds_split["test"]

targets = [sample['label'] for sample in ds_train]
class_sample_counts = np.bincount(targets)
weights = 1.0 / class_sample_counts
sample_weights = [weights[t] for t in targets]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

def custom_collate_fn(batch, transform):
    images, labels = [], []
    for item in batch:
        img = item['image']
        lbl = item['label']
        if lbl == 1 and transform == train_transforms and np.random.rand() > 0.5:
            img = augment_class1(img)
        else:
            img = transform(img)
        images.append(img)
        labels.append(lbl)
    
    return torch.stack(images), torch.tensor(labels)

train_loader = DataLoader(
    ds_train,
    batch_size=16,
    sampler=sampler,
    collate_fn=lambda b: custom_collate_fn(b, train_transforms)
)

val_loader = DataLoader(
    ds_val,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda b: custom_collate_fn(b, val_transforms)
)

# ------------------------
# MODELO VIT-B16
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Carrega o modelo ViT-B16 do torchvision
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# Congela todos os parâmetros por padrão
for param in model.parameters():
    param.requires_grad = False

# Libera o último bloco do encoder e o classificador (head)
for name, param in model.named_parameters():
    if "encoder.layers.encoder_layer_11" in name or "heads" in name:
        param.requires_grad = True

# O classificador do ViT já é ajustado automaticamente pela linha acima,
# mas podemos redefini-lo para maior clareza e controle
num_ftrs = model.heads.head.in_features
model.heads.head = nn.Linear(num_ftrs, 2)
model.to(device)

# ------------------------
# LOSS + OPTIMIZER
# ------------------------
labels_train = [sample['label'] for sample in ds_train]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_train),
    y=labels_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# ------------------------
# LOOP DE TREINO
# ------------------------
EPOCHS = 30
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(train_loader):.4f}")

# ------------------------
# AVALIAÇÃO - MATRIZ DE CONFUSÃO
# ------------------------
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Classe 0","Classe 1"], yticklabels=["Classe 0","Classe 1"])
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - ViT-B16")
plt.show()

# Treinamento com ResNet50 

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
from torch.utils.data import DataLoader
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Configurar o dispositivo (GPU ou CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

# 2. Carregar e dividir o dataset
ds = load_dataset("TimJaspersTue/RARE25-train", split='train')
ds_split = ds.train_test_split(test_size=0.2)
ds_train = ds_split['train']
ds_val = ds_split['test']

# 3. Definir as transformações de pré-processamento
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.RandomCrop(224, 224),
    A.SquareSymmetry(p=0.5),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# 4. **Função de Pré-processamento**
def preprocess_image(image, transforms):
    image_np = np.array(image.convert('RGB'))
    transformed = transforms(image=image_np)
    return transformed['image']

# 5. Criar uma função de colaboração (`collate_fn`)
def custom_collate_fn(batch):
    images = [preprocess_image(item['image'], train_transforms) for item in batch]
    labels = [item['label'] for item in batch]
    
    images = torch.stack(images, dim=0)
    labels = torch.tensor(labels, dtype=torch.long)
    
    return images, labels

# 6. Criar os DataLoaders
train_loader = DataLoader(ds_train, batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(ds_val, batch_size=16, collate_fn=custom_collate_fn)

# 7. Carregar o modelo do `torchvision` e ajustar
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model.to(device)

# 8. Definir a função de perda e o otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 9. Loop de treinamento
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    # 10. Loop de validação
    model.eval()
    total_val_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(ds_train)
    epoch_val_loss = total_val_loss / len(ds_val)
    accuracy = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, "
          f"Val Loss: {epoch_val_loss:.4f}, "
          f"Val Acc: {accuracy:.2f}%")

print("Treinamento concluído!")

In [ ]:
# 11. **Gera e exibe a matriz de confusão**
cm = confusion_matrix(all_labels, all_preds)
class_names = ['Classe 0', 'Classe 1']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Matriz de Confusão')
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import resnet50
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight

# ------------------------
# DATA AUGMENTATION
# ------------------------
train_transforms = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor()
])

augment_class1 = T.Compose([
    T.Resize((224, 224)),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAffine(degrees=15, translate=(0.1,0.1), scale=(0.9, 1.1)),
    T.RandomVerticalFlip(),
    T.ToTensor()
])

val_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

# ------------------------
# DATASET + SAMPLER
# ------------------------
try:
    ds_train
    ds_val
except NameError:
    ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
    ds_split = ds.train_test_split(test_size=0.2, seed=42)
    ds_train, ds_val = ds_split["train"], ds_split["test"]

targets = [sample['label'] for sample in ds_train]
class_sample_counts = np.bincount(targets)
weights = 1.0 / class_sample_counts
sample_weights = [weights[t] for t in targets]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

def custom_collate_fn(batch):
    images, labels = [], []
    for item in batch:
        img = item['image']
        lbl = item['label']
        # Aplica a augmentação extra só na classe 1
        if lbl == 1 and np.random.rand() > 0.5:
            img = augment_class1(img)
        else:
            img = train_transforms(img) # Assume transform de treino
        images.append(img)
        labels.append(lbl)
    return torch.stack(images), torch.tensor(labels)

def val_collate_fn(batch):
    images, labels = [], []
    for item in batch:
        img = val_transforms(item['image']) # Usa transform de validação
        lbl = item['label']
        images.append(img)
        labels.append(lbl)
    return torch.stack(images), torch.tensor(labels)

train_loader = DataLoader(
    ds_train,
    batch_size=16,
    sampler=sampler,
    collate_fn=custom_collate_fn
)

val_loader = DataLoader(
    ds_val,
    batch_size=16,
    shuffle=False,
    collate_fn=val_collate_fn
)
# ------------------------
# MODELO RESNET50
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet = resnet50(weights="IMAGENET1K_V2")

for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.layer4.parameters():
    param.requires_grad = True
for param in resnet.fc.parameters():
    param.requires_grad = True

resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet = resnet.to(device)

# ------------------------
# LOSS + OPTIMIZER
# ------------------------
class_weights = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-4)

# ------------------------
# LOOP DE TREINO
# ------------------------
EPOCHS = 30
best_acc = 0.0
for epoch in range(EPOCHS):
    resnet.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # --- validação ---
    resnet.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = resnet(imgs)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # --- métricas ---
    accuracy = np.mean(np.array(y_true) == np.array(y_pred)) * 100
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(train_loader):.4f} - Val Acc: {accuracy:.2f}%")
    
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(resnet.state_dict(), "best_resnet50.pth")
        print(">> Novo melhor modelo salvo!")

# --- AVALIAÇÃO FINAL ---
cm = confusion_matrix(y_true, y_pred)
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Classe 0","Classe 1"], yticklabels=["Classe 0","Classe 1"])
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - ResNet50")
plt.show()

# Swin T

In [ ]:
# Swin-Tiny otimizado • RARE25
# - WeightedRandomSampler
# - Augment extra só na classe 1
# - AMP (fp16)
# - Fine-tune em 2 estágios (topo -> unfreeze total)
# - Cosine scheduler, EarlyStopping (por F1 classe 1)
# - Threshold tuning final e matriz de confusão
# -------------------------------------------------------
import time, random, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

# Load default weights meta if available
# (torchvision provides mean/std in weight.meta for many models)
try:
    swin_w = models.Swin_T_Weights.DEFAULT
    MEAN, STD = swin_w.meta["mean"], swin_w.meta["std"]
except Exception:
    # fallback common normalization
    MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Base transforms
train_base = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# Extra augmentation only for positive (class 1)
train_pos_aug = T.Compose([
    T.Resize((224,224)),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    T.RandomAffine(degrees=12, translate=(0.08,0.08), scale=(0.95,1.05)),
    T.RandomVerticalFlip(p=0.25),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_tf = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def collate_fn(batch, transform, pos_transform=None, p_pos_aug=0.5, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = item["image"]
        lbl = int(item["label"])
        if is_train and lbl == 1 and pos_transform and random.random() < p_pos_aug:
            img = pos_transform(img)
        else:
            img = transform(img)
        imgs.append(img); labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders
BATCH = 16
train_loader = DataLoader(
    ds_train, batch_size=BATCH, sampler=sampler,
    collate_fn=lambda b: collate_fn(b, train_base, train_pos_aug, 0.5, True),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, val_tf, None, 0.0, False),
    num_workers=4, pin_memory=True
)

# -----------------------
# Modelo: Swin-Tiny (torchvision)
# -----------------------
model = models.swin_t(weights=getattr(models, "Swin_T_Weights", None) and models.Swin_T_Weights.DEFAULT)
# If weights object is not available, above will set model with default init

# Adjust classifier head for 2 classes
# torchvision swin has attribute `head` (nn.Linear)
try:
    in_f = model.head.in_features
    model.head = nn.Linear(in_f, 2)
except Exception:
    # fallback: try to find 'head' param
    for name, m in model.named_modules():
        if hasattr(m, 'in_features'):
            model.head = nn.Linear(m.in_features, 2)
            break

model.to(DEVICE)

# -----------------------
# Freeze all -> Stage1: unfreeze last stage + head
# -----------------------
for p in model.parameters():
    p.requires_grad = False

# In torchvision Swin, stages typically in named params 'layers.3' (stage index 3)
for name, p in model.named_parameters():
    if "layers.3" in name or "head" in name:
        p.requires_grad = True

# -----------------------
# Loss, Optimizer, Scheduler
# -----------------------
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cls_w_t)   # fast & effective

# optimizer stage1 -> only params with requires_grad True
params_stage1 = [p for p in model.parameters() if p.requires_grad]
opt = optim.AdamW(params_stage1, lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=8)

# AMP scaler
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def train_one_epoch(loader, model, opt, crit, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            loss = crit(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / max(1, n), elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs); all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

# -----------------------
# Training: Stage1 (top)
# -----------------------
E1 = 6   # curto para acelerar
E2 = 20  # stage2 epochs (total ~26) — ajuste conforme necessidade
best_f1 = -1.0
patience = 5
counter = 0
best_state = None

print("== STAGE 1: head + final stage ==")
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage1 (patience).")
        break

# -----------------------
# Stage2: unfreeze all (fine-tune completo) com lr menor
# -----------------------
print("\n== STAGE 2: unfreeze all (fine-tune completo) ==")
for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
        torch.save(best_state, "best_swin_tiny_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage2 (patience).")
        break

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f})")

# -----------------------
# Threshold tuning final e matriz de confusão
# -----------------------
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1); y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.15, 0.60, 92)
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th:
        best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f}  (F1={best_f1_th:.4f})")
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))

plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - Swin-Tiny"); plt.colorbar(); plt.show()


# Modelos finais

## Swin-Tiny otimizado • RARE25

In [ ]:
# Swin-Tiny otimizado • RARE25
# - WeightedRandomSampler
# - Augment extra só na classe 1
# - AMP (fp16)
# - Fine-tune em 2 estágios (topo -> unfreeze total)
# - Cosine scheduler, EarlyStopping (por F1 classe 1)
# - Threshold tuning final e matriz de confusão
# -------------------------------------------------------
import time, random, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

# Load default weights meta if available
# (torchvision provides mean/std in weight.meta for many models)
try:
    swin_w = models.Swin_T_Weights.DEFAULT
    MEAN, STD = swin_w.meta["mean"], swin_w.meta["std"]
except Exception:
    # fallback common normalization
    MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Base transforms
train_base = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# Extra augmentation only for positive (class 1)
train_pos_aug = T.Compose([
    T.Resize((224,224)),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    T.RandomAffine(degrees=12, translate=(0.08,0.08), scale=(0.95,1.05)),
    T.RandomVerticalFlip(p=0.25),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_tf = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def collate_fn(batch, transform, pos_transform=None, p_pos_aug=0.5, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = item["image"]
        lbl = int(item["label"])
        if is_train and lbl == 1 and pos_transform and random.random() < p_pos_aug:
            img = pos_transform(img)
        else:
            img = transform(img)
        imgs.append(img); labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders
BATCH = 16
train_loader = DataLoader(
    ds_train, batch_size=BATCH, sampler=sampler,
    collate_fn=lambda b: collate_fn(b, train_base, train_pos_aug, 0.5, True),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, val_tf, None, 0.0, False),
    num_workers=4, pin_memory=True
)

# -----------------------
# Modelo: Swin-Tiny (torchvision)
# -----------------------
model = models.swin_t(weights=getattr(models, "Swin_T_Weights", None) and models.Swin_T_Weights.DEFAULT)
# If weights object is not available, above will set model with default init

# Adjust classifier head for 2 classes
# torchvision swin has attribute `head` (nn.Linear)
try:
    in_f = model.head.in_features
    model.head = nn.Linear(in_f, 2)
except Exception:
    # fallback: try to find 'head' param
    for name, m in model.named_modules():
        if hasattr(m, 'in_features'):
            model.head = nn.Linear(m.in_features, 2)
            break

model.to(DEVICE)

# -----------------------
# Freeze all -> Stage1: unfreeze last stage + head
# -----------------------
for p in model.parameters():
    p.requires_grad = False

# In torchvision Swin, stages typically in named params 'layers.3' (stage index 3)
for name, p in model.named_parameters():
    if "layers.3" in name or "head" in name:
        p.requires_grad = True

# -----------------------
# Loss, Optimizer, Scheduler
# -----------------------
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cls_w_t)   # fast & effective

# optimizer stage1 -> only params with requires_grad True
params_stage1 = [p for p in model.parameters() if p.requires_grad]
opt = optim.AdamW(params_stage1, lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=8)

# AMP scaler
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def train_one_epoch(loader, model, opt, crit, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            loss = crit(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / max(1, n), elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs); all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

# -----------------------
# Training: Stage1 (top)
# -----------------------
E1 = 6   # curto para acelerar
E2 = 20  # stage2 epochs (total ~26) — ajuste conforme necessidade
best_f1 = -1.0
patience = 5
counter = 0
best_state = None

print("== STAGE 1: head + final stage ==")
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage1 (patience).")
        break

# -----------------------
# Stage2: unfreeze all (fine-tune completo) com lr menor
# -----------------------
print("\n== STAGE 2: unfreeze all (fine-tune completo) ==")
for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
        torch.save(best_state, "best_swin_tiny_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage2 (patience).")
        break

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f})")

# -----------------------
# Threshold tuning final e matriz de confusão
# -----------------------
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1); y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.15, 0.60, 92)
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th:
        best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f}  (F1={best_f1_th:.4f})")
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))

plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - Swin-Tiny"); plt.colorbar(); plt.show()


## ViT-B16 otimizado: AMP (fp16) + early stopping + fine-tune em 2 estágios

In [ ]:
# ViT-B16 otimizado: AMP (fp16) + early stopping + fine-tune em 2 estágios
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# -----------------------
# Config / Reprodutibilidade
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

# mean/std do ViT (weights meta)
vit_w = models.ViT_B_16_Weights.DEFAULT
# Access mean and std from the transforms method
MEAN = vit_w.transforms().mean
STD = vit_w.transforms().std

# base transforms (ambas classes)
# Base transforms
train_base = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# Extra augmentation only for positive (class 1)
train_pos_aug = T.Compose([
    T.Resize((224,224)),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    T.RandomAffine(degrees=12, translate=(0.08,0.08), scale=(0.95,1.05)),
    T.RandomVerticalFlip(p=0.25),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_tf = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def collate_fn(batch, transform, pos_transform=None, p_pos_aug=0.5, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = item["image"]
        lbl = int(item["label"])
        if is_train and lbl == 1 and pos_transform and random.random() < p_pos_aug:
            img = pos_transform(img)
        else:
            img = transform(img)
        imgs.append(img)
        labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders (batch menor obrigatório para ViT + AMP)
BATCH = 16
train_loader = DataLoader(ds_train, batch_size=BATCH, sampler=sampler,
                          collate_fn=lambda b: collate_fn(b, train_base, train_pos_aug, 0.5, True),
                          num_workers=4, pin_memory=True)
val_loader = DataLoader(ds_val, batch_size=BATCH, shuffle=False,
                        collate_fn=lambda b: collate_fn(b, val_tf, None, 0.0, False),
                        num_workers=4, pin_memory=True)

# -----------------------
# Modelo (ViT-B16)
# -----------------------
model = models.vit_b_16(weights=vit_w)
in_f = model.heads.head.in_features
model.heads.head = nn.Linear(in_f, 2)
model.to(DEVICE)

# congelar tudo
for p in model.parameters():
    p.requires_grad = False
# liberar último bloco + head (stage 1)
for name, p in model.named_parameters():
    if "encoder.layers.encoder_layer_11" in name or "heads" in name:
        p.requires_grad = True

# -----------------------
# Loss (use focal alternativo) - aqui usamos CrossEntropy com class weights para velocidade
# -----------------------
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cls_w_t)

# -----------------------
# Opt + scheduler
# -----------------------
# stage1 optimizer (somente params liberados)
params_stage1 = [p for p in model.parameters() if p.requires_grad]
opt = optim.AdamW(params_stage1, lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)

# -----------------------
# AMP & training helpers
# -----------------------
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def train_one_epoch(loader, model, opt, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / n, elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs)
            all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

# -----------------------
# Training: Stage1 (somente topo)
# -----------------------
E1 = 8   # reduzir stage1 pra acelerar
E2 = 22  # total ~30 (ajustável)
best_f1 = -1.0
patience = 6
counter = 0
best_state = None
best_epoch = 0

print("== STAGE 1: head + último bloco ==")
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    # validação rápida (threshold 0.5)
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    # early save by f1
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); best_epoch = ep; counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage1 (patience).")
        break

# -----------------------
# Unfreeze all (Stage2) + smaller lr + scheduler + continue training
# -----------------------
print("\n== STAGE 2: unfreeze total (fine-tune completo) ==")
for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); best_epoch = ("S2", ep); counter = 0
        torch.save(best_state, "best_vit_b16_fp16_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage2 (patience).")
        break

# restore best
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f}, epoch={best_epoch})")

# -----------------------
# Threshold tuning final (usa val probs para maximizar F1 classe 1)
# -----------------------
# coletar probs
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1)
        y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.2, 0.6, 81)
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th:
        best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f}  (F1={best_f1_th:.4f})")

# final confusion matrix
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))
plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - ViT-B16 (fp16 otimiz.)"); plt.colorbar(); plt.show()

## ResNet50 otimizado: AMP(fp16) + early stopping + fine-tune em 2 estagios

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import resnet50
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------
# DATA AUGMENTATION
# ------------------------
train_transforms = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor()
])

augment_class1 = T.Compose([
    T.Resize((224, 224)), # Added Resize here
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAffine(degrees=15, translate=(0.1,0.1), scale=(0.9, 1.1)),
    T.RandomVerticalFlip(),
    T.ToTensor()
])

def custom_collate_fn(batch, transform):
    images, labels = [], []
    for item in batch:
        img = item['image']
        lbl = item['label']
        if lbl == 1 and np.random.rand() > 0.5:  # augment extra só na classe 1
            img = augment_class1(img)
        else:
            img = transform(img)
        images.append(img)
        labels.append(lbl)
    return torch.stack(images), torch.tensor(labels)

# ------------------------
# DATASET + SAMPLER
# ------------------------
targets = [sample['label'] for sample in ds_train] # Corrected: access label by key

class_sample_counts = np.bincount(targets)
weights = 1.0 / class_sample_counts
sample_weights = [weights[t] for t in targets]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    ds_train,
    batch_size=16,
    sampler=sampler,
    collate_fn=lambda b: custom_collate_fn(b, train_transforms)
)

val_loader = DataLoader(
    ds_val,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda b: custom_collate_fn(b, train_transforms)
)

# ------------------------
# MODELO RESNET50
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet = resnet50(weights="IMAGENET1K_V2")

# Congela tudo
for param in resnet.parameters():
    param.requires_grad = False

# Libera último bloco + FC
for param in resnet.layer4.parameters():
    param.requires_grad = True
for param in resnet.fc.parameters():
    param.requires_grad = True

resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet = resnet.to(device)

# ------------------------
# LOSS + OPTIMIZER
# ------------------------
class_weights = torch.tensor(1.0 / class_sample_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-4)

# ------------------------
# LOOP DE TREINO
# ------------------------
EPOCHS = 30
resnet.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {total_loss/len(train_loader):.4f}")

# ------------------------
# AVALIAÇÃO - MATRIZ DE CONFUSÃO
# ------------------------
resnet.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = resnet(imgs)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Matriz de confusão
cm = confusion_matrix(y_true, y_pred)
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Classe 0","Classe 1"], yticklabels=["Classe 0","Classe 1"])
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - ResNet50")
plt.show()

# melhor modelo com shap

In [ ]:
!pip install shap

In [ ]:
# Swin-Tiny otimizado • RARE25
# - WeightedRandomSampler
# - Augment extra só na classe 1
# - AMP (fp16)
# - Fine-tune em 2 estágios (topo -> unfreeze total)
# - Cosine scheduler, EarlyStopping (por F1 classe 1)
# - Threshold tuning final e matriz de confusão
# -------------------------------------------------------
import time, random, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Biblioteca SHAP para interpretabilidade
try:
    import shap
except ImportError:
    print("A biblioteca SHAP não está instalada. Execute 'pip install shap' para usá-la.")
    exit()

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

try:
    swin_w = models.Swin_T_Weights.DEFAULT
    MEAN, STD = swin_w.meta["mean"], swin_w.meta["std"]
except Exception:
    MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_base = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_pos_aug = T.Compose([
    T.Resize((224,224)),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    T.RandomAffine(degrees=12, translate=(0.08,0.08), scale=(0.95,1.05)),
    T.RandomVerticalFlip(p=0.25),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_tf = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def collate_fn(batch, transform, pos_transform=None, p_pos_aug=0.5, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = item["image"]
        lbl = int(item["label"])
        if is_train and lbl == 1 and pos_transform and random.random() < p_pos_aug:
            img = pos_transform(img)
        else:
            img = transform(img)
        imgs.append(img); labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH = 16
train_loader = DataLoader(
    ds_train, batch_size=BATCH, sampler=sampler,
    collate_fn=lambda b: collate_fn(b, train_base, train_pos_aug, 0.5, True),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, val_tf, None, 0.0, False),
    num_workers=4, pin_memory=True
)

# -----------------------
# Modelo: Swin-Tiny (torchvision)
# -----------------------
model = models.swin_t(weights=getattr(models, "Swin_T_Weights", None) and models.Swin_T_Weights.DEFAULT)
try:
    in_f = model.head.in_features
    model.head = nn.Linear(in_f, 2)
except Exception:
    for name, m in model.named_modules():
        if hasattr(m, 'in_features'):
            model.head = nn.Linear(m.in_features, 2)
            break
model.to(DEVICE)

# -----------------------
# Treinamento
# -----------------------
def train_one_epoch(loader, model, opt, crit, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            loss = crit(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / max(1, n), elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs); all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

E1 = 6; E2 = 20
best_f1 = -1.0
patience = 5
counter = 0
best_state = None

print("== STAGE 1: head + final stage ==")
for p in model.parameters(): p.requires_grad = False
for name, p in model.named_parameters():
    if "layers.3" in name or "head" in name: p.requires_grad = True
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cls_w_t)
opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E1)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f} f1_val={f1_val:.4f} time={t:.1f}s lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1: best_f1 = f1_val; best_state = model.state_dict(); counter = 0
    else: counter += 1
    if counter >= patience: print("Early stopping stage1 (patience)."); break

print("\n== STAGE 2: unfreeze all (fine-tune completo) ==")
for p in model.parameters(): p.requires_grad = True
opt = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))
counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f} f1_val={f1_val:.4f} time={t:.1f}s lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
        torch.save(best_state, "best_swin_tiny_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else: counter += 1
    if counter >= patience: print("Early stopping stage2 (patience)."); break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f})")

# -----------------------
# Threshold tuning final e matriz de confusão
# -----------------------
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1); y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.15, 0.60, 92)
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th: best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f} (F1={best_f1_th:.4f})")
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))

plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm): plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - Swin-Tiny"); plt.colorbar(); plt.show()

# -------------------------------------------------------
# Aplicação do Método SHAP
# -------------------------------------------------------
print("\n== Análise de Interpretabilidade com SHAP ==")

# 1. Defina um conjunto de dados de fundo para o SHAP
# Uma amostra de 50-100 imagens do conjunto de treinamento é recomendada
BACKGROUND_SIZE = 50
background_data_loader = DataLoader(
    ds_train, batch_size=BACKGROUND_SIZE, shuffle=True,
    collate_fn=lambda b: collate_fn(b, val_tf, None, 0.0, False)
)
background_data, _ = next(iter(background_data_loader))
background_data = background_data.to(DEVICE)

# 2. Crie o explainer SHAP
# O DeepExplainer é o mais indicado para modelos de redes neurais profundas como o Swin-Tiny.
explainer = shap.DeepExplainer(model, background_data)

# 3. Pegue um batch de imagens de validação para análise
sample_batch, sample_labels = next(iter(val_loader))
sample_batch = sample_batch.to(DEVICE)
sample_labels = sample_labels.numpy()

# 4. Calcule os valores de SHAP para o batch
print("Calculando os valores de SHAP (isso pode demorar um pouco)...")
shap_values = explainer.shap_values(sample_batch)

# 5. Desnormalize as imagens para a visualização
mean_tensor = torch.tensor(MEAN, device=DEVICE).view(3, 1, 1)
std_tensor = torch.tensor(STD, device=DEVICE).view(3, 1, 1)
sample_batch_unnormalized = (sample_batch * std_tensor) + mean_tensor
sample_batch_unnormalized = sample_batch_unnormalized.permute(0, 2, 3, 1).cpu().numpy()

# 6. Plote as explicações SHAP para a Classe 1 (posição 1 na lista)
print("\nExibindo explicações SHAP para a Classe 1 (vermelho = contribuição positiva)")
# A função image_plot precisa dos valores SHAP e das imagens de entrada
shap.image_plot(shap_values[1], sample_batch_unnormalized, labels=sample_labels)

# Você pode inspecionar a primeira imagem individualmente para ver a previsão
# e os valores de SHAP
# pred_logits = model(sample dim=1)[:, 1].item()
# print(f"\nAnalisando a primeira imagem. Classe v_batch[0].unsqueeze(0))
# pred_prob = torch.softmax(pred_logits,erdadeira: {sample_labels[0]}, Probabilidade predita (Classe 1): {pred_prob:.4f}")

# Modelo para o open development test

In [ ]:
# Swin-Tiny Otimizado - RARE25 (Versão Albumentations)
# -------------------------------------------------------
import time, random, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# NOVO: Instale o Albumentations com pip install albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---- Implementação da Focal Loss (CORRIGIDA) ----
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets, class_weights=None):
        # nn.CrossEntropyLoss lida com os pesos de classe
        BCE_loss = nn.CrossEntropyLoss(weight=class_weights, reduction='none')(inputs, targets)
        
        pt = torch.exp(-BCE_loss)
        F_loss = (1 - pt)**self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms (ADAPTAÇÃO PARA ALBUMENTATIONS)
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

# Normalização padrão
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Extra augmentation unificada para Albumentations
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.25),
    A.Rotate(limit=10),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.OneOf([A.GaussNoise(), A.ISONoise()], p=0.2),
    A.RandomRain(p=0.1),
    A.SquareSymmetry(p=0.5),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# NOVO: collate_fn adaptada para Albumentations
def collate_fn(batch, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = np.array(item["image"]) # Albumentations precisa de um numpy array
        lbl = int(item["label"])
        
        if is_train:
            augmented = train_transforms(image=img)
            img = augmented['image']
        else:
            augmented = val_transforms(image=img)
            img = augmented['image']
            
        imgs.append(img); labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado (mantido)
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders (ADAPTAÇÃO)
BATCH = 16
train_loader = DataLoader(
    ds_train, batch_size=BATCH, sampler=sampler,
    collate_fn=lambda b: collate_fn(b, is_train=True),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, is_train=False),
    num_workers=4, pin_memory=True
)

# -----------------------
# Modelo: Swin-Tiny (mantido)
# -----------------------
model = models.swin_t(weights=getattr(models, "Swin_T_Weights", None) and models.Swin_T_Weights.DEFAULT)

try:
    in_f = model.head.in_features
    model.head = nn.Linear(in_f, 2)
except Exception:
    for name, m in model.named_modules():
        if hasattr(m, 'in_features'):
            model.head = nn.Linear(m.in_features, 2)
            break

model.to(DEVICE)

# -----------------------
# Freeze all -> Stage1: unfreeze last stage + head (mantido)
# -----------------------
for p in model.parameters():
    p.requires_grad = False

for name, p in model.named_parameters():
    if "layers.3" in name or "head" in name:
        p.requires_grad = True

# -----------------------
# Loss, Optimizer, Scheduler (CORRIGIDO)
# -----------------------
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)

# Inicializa FocalLoss sem o alpha
criterion = FocalLoss(gamma=2)

params_stage1 = [p for p in model.parameters() if p.requires_grad]
opt = optim.AdamW(params_stage1, lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def train_one_epoch(loader, model, opt, crit, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            # Passa o tensor de pesos de classe para o critério
            loss = crit(logits, y, class_weights=cls_w_t)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / max(1, n), elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs); all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

# -----------------------
# Training: Stage1 (top) (mantido)
# -----------------------
E1 = 10
E2 = 25
best_f1 = -1.0
patience = 8
counter = 0
best_state = None

print("== STAGE 1: head + final stage ==")
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage1 (patience).")
        break

# -----------------------
# Stage2: unfreeze all (fine-tune completo) com lr menor (mantido)
# -----------------------
print("\n== STAGE 2: unfreeze all (fine-tune completo) ==")
for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
        torch.save(best_state, "best_swin_tiny_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage2 (patience).")
        break

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f})")

# -----------------------
# Threshold tuning final e matriz de confusão (mantido)
# -----------------------
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1); y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.05, 0.95, 100) 
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th:
        best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f}  (F1={best_f1_th:.4f})")
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))

plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - Swin-Tiny"); plt.colorbar(); plt.show()

## Tentativa 3 
FocalLoss corrigida para lidar com o desequilíbrio de classes.

Uma pipeline de Albumentations mais robusta e corretamente normalizada.

Uma camada de Dropout adicionada para combater o overfitting.

Taxas de aprendizado e paciência de EarlyStopping ajustadas.

In [ ]:
# Swin-Tiny Otimizado - RARE25 (Versão Albumentations com Dropout)
#nao foi enviado
# -------------------------------------------------------
import time, random, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# NOVO: Instale o Albumentations com pip install albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---- Implementação da Focal Loss (CORRIGIDA) ----
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets, class_weights=None):
        BCE_loss = nn.CrossEntropyLoss(weight=class_weights, reduction='none')(inputs, targets)
        
        pt = torch.exp(-BCE_loss)
        F_loss = (1 - pt)**self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms (ADAPTAÇÃO PARA ALBUMENTATIONS)
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

# Normalização padrão
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Extra augmentation unificada para Albumentations
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.25),
    A.Rotate(limit=10),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.OneOf([A.GaussNoise(), A.ISONoise()], p=0.2),
    A.RandomRain(p=0.1),
    A.SquareSymmetry(p=0.5),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# NOVO: collate_fn adaptada para Albumentations
def collate_fn(batch, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = np.array(item["image"]) 
        lbl = int(item["label"])
        
        if is_train:
            augmented = train_transforms(image=img)
            img = augmented['image']
        else:
            augmented = val_transforms(image=img)
            img = augmented['image']
            
        imgs.append(img); labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado (mantido)
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders (ADAPTAÇÃO)
BATCH = 16
train_loader = DataLoader(
    ds_train, batch_size=BATCH, sampler=sampler,
    collate_fn=lambda b: collate_fn(b, is_train=True),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, is_train=False),
    num_workers=4, pin_memory=True
)

# -----------------------
# Modelo: Swin-Tiny (COM DROPOUT ADICIONAL)
# -----------------------
model = models.swin_t(weights=getattr(models, "Swin_T_Weights", None) and models.Swin_T_Weights.DEFAULT)

try:
    in_f = model.head.in_features
    # Adicione uma camada de Dropout com uma taxa de 20%
    model.head = nn.Sequential(
        nn.Dropout(p=0.2), 
        nn.Linear(in_f, 2)
    )
except Exception:
    for name, m in model.named_modules():
        if hasattr(m, 'in_features'):
            # Adicione a camada de Dropout na camada de fallback também
            model.head = nn.Sequential(
                nn.Dropout(p=0.2),
                nn.Linear(m.in_features, 2)
            )
            break

model.to(DEVICE)

# -----------------------
# Freeze all -> Stage1: unfreeze last stage + head (mantido)
# -----------------------
for p in model.parameters():
    p.requires_grad = False

for name, p in model.named_parameters():
    if "layers.3" in name or "head" in name:
        p.requires_grad = True

# -----------------------
# Loss, Optimizer, Scheduler (CORRIGIDO)
# -----------------------
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)

# Inicializa FocalLoss sem o alpha
criterion = FocalLoss(gamma=2)

params_stage1 = [p for p in model.parameters() if p.requires_grad]
opt = optim.AdamW(params_stage1, lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def train_one_epoch(loader, model, opt, crit, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            loss = crit(logits, y, class_weights=cls_w_t)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / max(1, n), elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs); all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

# -----------------------
# Training: Stage1 (top) (mantido)
# -----------------------
E1 = 10
E2 = 25
best_f1 = -1.0
patience = 8
counter = 0
best_state = None

print("== STAGE 1: head + final stage ==")
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage1 (patience).")
        break

# -----------------------
# Stage2: unfreeze all (fine-tune completo) com lr menor (mantido)
# -----------------------
print("\n== STAGE 2: unfreeze all (fine-tune completo) ==")
for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
        torch.save(best_state, "best_swin_tiny_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage2 (patience).")
        break

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f})")

# -----------------------
# Threshold tuning final e matriz de confusão (mantido)
# -----------------------
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1); y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.05, 0.95, 100) 
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th:
        best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f}  (F1={best_f1_th:.4f})")
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))

plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - Swin-Tiny"); plt.colorbar(); plt.show()

# Modelo com os pesos do gastroVision

In [ ]:
# Swin-Tiny Otimizado - RARE25 (Versão Albumentations)
# -------------------------------------------------------
import time, random, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# NOVO: Instale o Albumentations com pip install albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---- Implementação da Focal Loss (CORRIGIDA) ----
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets, class_weights=None):
        BCE_loss = nn.CrossEntropyLoss(weight=class_weights, reduction='none')(inputs, targets)
        
        pt = torch.exp(-BCE_loss)
        F_loss = (1 - pt)**self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

# -----------------------
# Config
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Dataset + transforms (ADAPTAÇÃO PARA ALBUMENTATIONS)
# -----------------------
ds = load_dataset("TimJaspersTue/RARE25-train", split="train")
split = ds.train_test_split(test_size=0.2, seed=SEED)
ds_train, ds_val = split["train"], split["test"]

# Normalização padrão
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Extra augmentation unificada para Albumentations
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.25),
    A.Rotate(limit=10),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.OneOf([A.GaussNoise(), A.ISONoise()], p=0.2),
    A.RandomRain(p=0.1),
    A.SquareSymmetry(p=0.5),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# NOVO: collate_fn adaptada para Albumentations
def collate_fn(batch, is_train=True):
    imgs, labels = [], []
    for item in batch:
        img = np.array(item["image"])
        lbl = int(item["label"])
        
        if is_train:
            augmented = train_transforms(image=img)
            img = augmented['image']
        else:
            augmented = val_transforms(image=img)
            img = augmented['image']
            
        imgs.append(img); labels.append(lbl)
    return torch.stack(imgs), torch.tensor(labels, dtype=torch.long)

# -----------------------
# Sampler balanceado (mantido)
# -----------------------
targets = [int(x["label"]) for x in ds_train]
class_counts = np.bincount(targets)
print("Class counts (train):", class_counts)
inv_weights = 1.0 / class_counts
sample_weights = [inv_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders (ADAPTAÇÃO)
BATCH = 16
train_loader = DataLoader(
    ds_train, batch_size=BATCH, sampler=sampler,
    collate_fn=lambda b: collate_fn(b, is_train=True),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    ds_val, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, is_train=False),
    num_workers=4, pin_memory=True
)

# -----------------------
# Modelo: Swin-Tiny (CORRIGIDO PARA O FINE-TUNING FINAL)
# -----------------------
# Inicie o modelo sem pesos pré-treinados
model = models.swin_t(weights=None) 

# PASSO 1: Ajuste a camada head para 27 classes (o que o GastroVision espera)
try:
    in_f = model.head.in_features
    model.head = nn.Linear(in_f, 27)
except Exception:
    for name, m in model.named_modules():
        if hasattr(m, 'in_features'):
            model.head = nn.Linear(m.in_features, 27)
            break

model.to(DEVICE)

# PASSO 2: Carregue os pesos do modelo GastroVision
model.load_state_dict(torch.load("/kaggle/input/gastrovisionswin/pytorch/default/2/swin_gastrovision (1).pth"))

# PASSO 3: Agora, ajuste a camada head para a tarefa final do RARE25 (2 classes)
model.head = nn.Linear(in_f, 2)
model.to(DEVICE)

# -----------------------
# Freeze all -> Stage1: unfreeze last stage + head (mantido)
# -----------------------
for p in model.parameters():
    p.requires_grad = False

for name, p in model.named_parameters():
    if "layers.3" in name or "head" in name:
        p.requires_grad = True

# -----------------------
# Loss, Optimizer, Scheduler (CORRIGIDO)
# -----------------------
cls_w = compute_class_weight("balanced", classes=np.unique(targets), y=targets)
cls_w_t = torch.tensor(cls_w, dtype=torch.float32).to(DEVICE)

# Inicializa FocalLoss sem o alpha
criterion = FocalLoss(gamma=2)

params_stage1 = [p for p in model.parameters() if p.requires_grad]
opt = optim.AdamW(params_stage1, lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def train_one_epoch(loader, model, opt, crit, scaler, device):
    model.train()
    running_loss = 0.0
    n = 0
    t0 = time.time()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(x)
            loss = crit(logits, y, class_weights=cls_w_t)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    elapsed = time.time() - t0
    return running_loss / max(1, n), elapsed

def validate(loader, model, device, threshold=0.5):
    model.eval()
    all_y, all_p1 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_p1.extend(probs); all_y.extend(y.numpy())
    all_y = np.array(all_y); all_p1 = np.array(all_p1)
    preds = (all_p1 > threshold).astype(int)
    f1 = f1_score(all_y, preds, pos_label=1)
    return f1, all_y, all_p1

# -----------------------
# Training: Stage1 (top) (mantido)
# -----------------------
E1 = 10
E2 = 25
best_f1 = -1.0
patience = 8
counter = 0
best_state = None

print("== STAGE 1: head + final stage ==")
for ep in range(1, E1+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, _, _ = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S1 {ep}/{E1}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage1 (patience).")
        break

# -----------------------
# Stage2: unfreeze all (fine-tune completo) com lr menor (mantido)
# -----------------------
print("\n== STAGE 2: unfreeze all (fine-tune completo) ==")
for p in model.parameters():
    p.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E2)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

counter = 0
for ep in range(1, E2+1):
    loss, t = train_one_epoch(train_loader, model, opt, criterion, scaler, DEVICE)
    sched.step()
    f1_val, y_true, y_prob1 = validate(val_loader, model, DEVICE, threshold=0.5)
    print(f"[S2 {ep}/{E2}] loss={loss:.4f}  f1_val={f1_val:.4f}  time={t:.1f}s  lr={sched.get_last_lr()[0]:.2e}")
    if f1_val > best_f1:
        best_f1 = f1_val; best_state = model.state_dict(); counter = 0
        torch.save(best_state, "best_swin_tiny_rare25.pth")
        print("  -> Novo melhor modelo salvo.")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping stage2 (patience).")
        break

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modelo restaurado do melhor checkpoint (f1={best_f1:.4f})")

# -----------------------
# Threshold tuning final e matriz de confusão (mantido)
# -----------------------
model.eval()
y_true, y_prob1 = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        logits = model(x)
        p1 = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        y_prob1.extend(p1); y_true.extend(y.numpy())
y_true = np.array(y_true); y_prob1 = np.array(y_prob1)

ths = np.linspace(0.05, 0.95, 100) 
best_th, best_f1_th = 0.5, -1
for th in ths:
    preds = (y_prob1 > th).astype(int)
    f1 = f1_score(y_true, preds, pos_label=1)
    if f1 > best_f1_th:
        best_f1_th = f1; best_th = th

print(f"\nThreshold ótimo para F1 classe 1: {best_th:.3f}  (F1={best_f1_th:.4f})")
y_pred_final = (y_prob1 > best_th).astype(int)
cm = confusion_matrix(y_true, y_pred_final)
print("\nClassification Report (threshold ajustado):\n", classification_report(y_true, y_pred_final, digits=4, target_names=["Classe 0","Classe 1"]))

plt.figure(figsize=(5,4)); plt.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.xticks([0,1], ["Classe 0","Classe 1"]); plt.yticks([0,1], ["Classe 0","Classe 1"])
plt.title("Matriz de Confusão - Swin-Tiny"); plt.colorbar(); plt.show()